# 02: Preprocessing, Diabetes 130-US Hospitals Dataset

Transforms the raw UCI Diabetes 130-US Hospitals file (Strack et al.,
2014) into the analysis dataset, binary target (any
readmission = 1, base rate ~46%), and fairness attribute `sex_female`.

Raw input: s3://osilesi-dissertation-data/raw/diabetic_data.csv
Output: s3://osilesi-dissertation-data/processed/health_final_pruned.csv

In [1]:
import pandas as pd
import numpy as np

BUCKET = "osilesi-dissertation-data-2026"   # bucket name
pd.set_option("display.max_columns", None)

In [2]:
diab = pd.read_csv(f"s3://{BUCKET}/raw/diabetic_data.csv")
print(diab.shape)    # expect (101766, 50)

(101766, 50)


In [3]:
miss = (diab == "?").sum()
print(miss[miss > 0].sort_values(ascending=False))
# weight ~97%, medical_specialty ~49%, payer_code ~40%, plus race and diagnosis codes

weight               98569
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64


## Drop decisions, Diabetes

- weight: ~97% missing, 
- payer_code: ~40% missing,
- medical_specialty: ~49% missing, 
- encounter_id, patient_nbr: row identifiers, no predictive content
- 23 individual medication columns (metformin ... metformin-pioglitazone):
  not required for this study


In [4]:
diab = diab.drop(columns=["weight", "payer_code", "medical_specialty",
                          "encounter_id", "patient_nbr"])

In [5]:
med_cols = diab.loc[:, "metformin":"metformin-pioglitazone"].columns.tolist()
print(len(med_cols), "medication columns")   # ~23
diab = diab.drop(columns=med_cols)

23 medication columns


## Target construction

Binary readmission: both "<30" and ">30" map to 1 (any readmission);
"NO" maps to 0. Base rate ~46% positive at full data; experimental
class ratios are imposed downstream by apply_imbalance, not here.

In [6]:
diab["target"] = (diab["readmitted"] != "NO").astype(int)
diab = diab.drop(columns=["readmitted"])
print(diab["target"].value_counts(normalize=True))   # ~0.46 positive

target
0    0.539119
1    0.460881
Name: proportion, dtype: float64


## Gender handling

The raw gender column contains a small number of "Unknown/Invalid"
values. Rows should not be dropped at this juncture, so sex_female = 1 for "Female" and 0 otherwise; Unknown/Invalid
rows fall into the non-female group. 


In [7]:
print(diab["gender"].value_counts())
diab["sex_female"] = (diab["gender"] == "Female").astype(int)
diab = diab.drop(columns=["gender"])

gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64


## Rare diagnosis code consolidation

Implements self-imposed rule that diagnosis codes appearing in <1% of records are removed: rare codes in diag_1/diag_2/diag_3 are grouped into "Other" before encoding, so every resulting indicator column clears the 1% floor.


In [8]:
diab = diab.replace("?", "Unknown")

for c in ["diag_1", "diag_2", "diag_3"]:
    freq = diab[c].value_counts(normalize=True)
    rare = freq[freq < 0.01].index
    diab[c] = diab[c].where(~diab[c].isin(rare), "Other")
    print(c, "levels kept:", diab[c].nunique())

diag_1 levels kept: 24
diag_2 levels kept: 25
diag_3 levels kept: 21


In [9]:
cat_cols = diab.select_dtypes(include="object").columns.tolist()
print("Encoding:", cat_cols)
diab = pd.get_dummies(diab, columns=cat_cols, dtype=int)

n_features = diab.shape[1] - 1
assert n_features == 108, f"Feature count off: {n_features}"
assert diab.shape[0] == 101766, f"Row count off: {diab.shape[0]}"
assert diab.isna().sum().sum() == 0
assert "sex_female" in diab.columns
print(f"Features: {n_features} (Chapter Three states 108)")

Encoding: ['race', 'age', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed']
Features: 108 (Chapter Three states 108)


## Validation gate: PASSED [2026-08-14]

101,766 rows; 108 features (excl. target); 0 missing values; sex_female
present; positive class ~46%. 
Output written to /processed.

In [11]:
diab.to_csv(f"s3://{BUCKET}/processed/health_final_pruned.csv", index=False)